# Marine 48h Forecast — Single iTransformer, All 24 Parameters In & Out

A standalone deep-dive into **one model only**: iTransformer (Liu et al., 2023), which
took all 24 directly-modeled real parameters as input tokens and predicted all 24 as
output in the 11-model comparison (`Marine_Forecast_RealEMS_31Param.ipynb`) — there it
won outright on 12 of 24 parameters and posted the highest median skill (+75.3%) of any
model, by a wide margin. This notebook isolates it for closer inspection:

1. **Same data, same preprocessing** as the 11-model notebook (10-min resampling,
   6-duplicate collapse, 3 circular encodings, 2-day lookback / 48h horizon) — reuses the
   same cached `ems_10min_resampled.csv`.
2. **The 6 duplicate parameters get their own reconstruction plots** — actual vs.
   reconstructed (via the fitted linear formula applied to this model's forecast of the
   kept twin), not just a one-line MAE summary.
3. **Cross-variate attention weights are extracted and visualized** — iTransformer's
   distinguishing feature is that it attends *across parameters* (not across time), so
   the learned attention matrix directly answers "which parameters does the model rely
   on when forecasting this one?" — something none of the other 10 models in the main
   comparison can show.

This notebook is fully independent: it does not modify or depend on
`Marine_Forecast_RealEMS_31Param.ipynb`, `app.py`, or `app_realdata.py`.

## 0. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cpu")
torch.set_num_threads(8)   # see Marine_Forecast_RealEMS_31Param.ipynb Setup cell for why

print("PyTorch:", torch.__version__, "| torch threads:", torch.get_num_threads())


## 1. Load real EMS data (cached, 10-min resolution)

In [ ]:
df_10min = pd.read_csv("ems_10min_resampled.csv", index_col=0, parse_dates=True)
print(f"Loaded: {df_10min.shape[0]} rows x {df_10min.shape[1]} columns, "
      f"{df_10min.index.min()} -> {df_10min.index.max()}")

DUPLICATES = [
    ("airTemperature", "windChillTemperature"),
    ("tideLevel", "tidePressure"),
    ("tideLevel", "waterPressure"),
    ("tideLevel", "waterLevel"),
    ("waterTemperature", "waterTemperature_WQ"),
    ("significantWaveHeight", "maxWaveHeight"),
]
df_cat = df_10min[["precipitationType"]].copy()
df_num = df_10min.drop(columns=["precipitationType"]).copy()


## 2. Collapse duplicates, encode circular parameters

Same preprocessing as the 11-model notebook: 6 duplicate columns dropped (reconstructed
later from their kept twin via a fitted linear formula), 3 circular parameters
(`windDirection`, `currentDirection`, `compass`) sin/cos-encoded.

In [ ]:
CIRCULAR = ["windDirection", "currentDirection", "compass"]
for c in CIRCULAR:
    rad = np.deg2rad(df_num[c])
    df_num[f"{c}_sin"] = np.sin(rad)
    df_num[f"{c}_cos"] = np.cos(rad)
df_num_full = df_num.drop(columns=CIRCULAR)

target_cols = [c for c in df_num_full.columns if c not in [d for _, d in DUPLICATES]]
n_targets = len(target_cols)
print(f"Modeled channels (input AND output): {n_targets}")
print(target_cols)


## 3. Train/test split, duplicate reconstruction fit, scaling

Identical design to the 11-model notebook: lookback = 2 days (288 steps), horizon = 48
hours (288 steps), last 48h held out for validation.

In [ ]:
LOOKBACK, HORIZON = 288, 288   # 2 days lookback, 48h horizon @ 10-min steps

idx = df_num_full.index
df_num_full["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
df_num_full["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
df_num_full["dom_sin"] = np.sin(2 * np.pi * idx.day / 30)
df_num_full["dom_cos"] = np.cos(2 * np.pi * idx.day / 30)
calendar_cols = ["hour_sin", "hour_cos", "dom_sin", "dom_cos"]

feature_cols = target_cols + calendar_cols
model_data = df_num_full[feature_cols].copy()
n_features = len(feature_cols)
target_idx = [feature_cols.index(c) for c in target_cols]
calendar_idx = [feature_cols.index(c) for c in calendar_cols]

train_df = model_data.iloc[:-HORIZON].copy()
test_df = model_data.iloc[-HORIZON:].copy()
mean, std = train_df.mean(), train_df.std().replace(0, 1)
train_scaled = (train_df - mean) / std
full_scaled = (model_data - mean) / std

print(f"Targets: {n_targets}  |  Features (targets+calendar, i.e. iTransformer tokens): {n_features}")
print(f"Train: {train_df.shape[0]} rows ({train_df.shape[0]/144:.1f} days)")
print(f"Test : {test_df.shape[0]} rows ({test_df.shape[0]/144:.1f} days)  <- held-out validation")


In [ ]:
recon_coef = {}
for keep, drop in DUPLICATES:
    x, y = train_df[keep].values, df_num_full[drop].iloc[:-HORIZON].values
    slope, intercept = np.polyfit(x, y, 1)
    pred_train = slope * x + intercept
    r2 = 1 - np.sum((y - pred_train) ** 2) / np.sum((y - y.mean()) ** 2)
    recon_coef[drop] = (keep, float(slope), float(intercept), float(r2))
    print(f"  reconstruct {drop:25s} = {slope:.4f} * {keep} + {intercept:.4f}   (train R^2={r2:.5f})")


## 4. Build direct multi-horizon training windows

In [ ]:
def make_direct_windows(scaled_df, lookback, horizon, target_cols_idx):
    arr = scaled_df.values.astype(np.float32)
    X, Y = [], []
    for origin in range(lookback, len(arr) - horizon):
        X.append(arr[origin - lookback:origin])
        Y.append(arr[origin:origin + horizon][:, target_cols_idx])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X_direct, Y_direct = make_direct_windows(train_scaled, LOOKBACK, HORIZON, target_idx)
X_t, Y_t = torch.from_numpy(X_direct), torch.from_numpy(Y_direct)
n_val = max(1, int(0.1 * len(X_t)))
X_tr, Y_tr = X_t[:-n_val], Y_t[:-n_val]
X_val, Y_val = X_t[-n_val:], Y_t[-n_val:]
last_window = torch.from_numpy(train_scaled.values[-LOOKBACK:].astype(np.float32)).unsqueeze(0)
print(f"Training windows: {len(X_tr)} train / {len(X_val)} val")


## 5. The iTransformer model

Identical architecture to the 11-model notebook's Model 9 — each of the `n_features`
parameters (targets + calendar) becomes one token (its whole 2-day lookback window
embedded into a vector); self-attention runs **across tokens** (i.e. across
*parameters*, not across time), directly learning which parameters inform which.

In [ ]:
class ITransformer(nn.Module):
    def __init__(self, lookback, n_features, horizon, target_idx, d_model=64, n_heads=4,
                 n_layers=2, dropout=0.1):
        super().__init__()
        self.target_idx = target_idx
        self.embed = nn.Linear(lookback, d_model)
        self.var_id = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model * 2,
                                            dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        tok = self.embed(x.transpose(1, 2)) + self.var_id.unsqueeze(0)
        tok = self.encoder(tok)
        out = self.head(tok)
        return out.transpose(1, 2)[:, :, self.target_idx]

    def embed_tokens(self, x):
        # Just the tokenization step (embed + variate id), exposed for attention extraction.
        return self.embed(x.transpose(1, 2)) + self.var_id.unsqueeze(0)


def train_model(model, X_tr, Y_tr, X_val, Y_val, epochs=150, batch_size=64, lr=1e-3,
                 patience=20, name=""):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=6)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait = float("inf"), None, 0
    n = len(X_tr); t0 = time.time()
    for ep in range(epochs):
        ep_t0 = time.time()
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            xb, yb = X_tr[b].to(device), Y_tr[b].to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val.to(device)), Y_val.to(device)).item()
        sched.step(val_loss)
        print(f"  [{name}] epoch {ep+1:3d}/{epochs}  val_loss={val_loss:.4f}  "
              f"epoch_time={time.time()-ep_t0:.1f}s  elapsed={time.time()-t0:.0f}s", flush=True)
        if val_loss < best_val - 1e-5:
            best_val, wait = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    model.eval()
    print(f"{name:14s} best_val_loss={best_val:.4f}  epochs_run={ep+1:3d}  time={time.time()-t0:5.1f}s")
    return model

itransformer = ITransformer(LOOKBACK, n_features, HORIZON, target_idx, d_model=64, n_heads=4, n_layers=2)
itransformer = train_model(itransformer, X_tr, Y_tr, X_val, Y_val, epochs=150, patience=20, name="iTransformer")


## 6. Forecast all 24 parameters

In [ ]:
with torch.no_grad():
    pred_scaled = itransformer(last_window.to(device))[0].cpu().numpy()
preds_real = pred_scaled * std[target_cols].values + mean[target_cols].values
pred_df = pd.DataFrame(preds_real, columns=target_cols, index=test_df.index)
print("iTransformer 48h forecast complete (all 24 parameters, single model).")


## 7. Reconstruct circular parameters and score

Three circular pairs (wind direction, current direction, compass) reconstructed back to
degrees via `atan2(sin, cos)`.

In [ ]:
def reconstruct(pred_df_in):
    out = pred_df_in.copy()
    for ang in ["windDirection", "currentDirection", "compass"]:
        out[ang] = (np.rad2deg(np.arctan2(out[f"{ang}_sin"], out[f"{ang}_cos"])) % 360)
    return out

pred_final = reconstruct(pred_df)
truth = df_num_full.iloc[-HORIZON:].copy()
for ang in ["windDirection", "currentDirection", "compass"]:
    truth[ang] = (np.rad2deg(np.arctan2(truth[f"{ang}_sin"], truth[f"{ang}_cos"])) % 360)

report_params = [c for c in target_cols if not c.endswith(("_sin", "_cos"))] + \
                ["windDirection", "currentDirection", "compass"]
CIRCULAR_PARAMS = {"windDirection", "currentDirection", "compass"}

def circ_mae(true, pred):
    return np.abs((true - pred + 180) % 360 - 180).mean()

last_obs = df_num_full.iloc[-HORIZON - 1]
for ang in ["windDirection", "currentDirection", "compass"]:
    last_obs[ang] = (np.rad2deg(np.arctan2(last_obs[f"{ang}_sin"], last_obs[f"{ang}_cos"])) % 360)

metrics = []
for p in report_params:
    yt = truth[p].values
    yp_persist = np.repeat(last_obs[p], HORIZON)
    is_circular = p in CIRCULAR_PARAMS
    mae_p = circ_mae(yt, yp_persist) if is_circular else mean_absolute_error(yt, yp_persist)
    yhat = pred_final[p].values
    if is_circular:
        mae, rmse = circ_mae(yt, yhat), np.nan
    else:
        mae = mean_absolute_error(yt, yhat)
        rmse = np.sqrt(mean_squared_error(yt, yhat))
    skill = (1 - mae / mae_p) * 100 if mae_p > 0 else np.nan
    metrics.append({"parameter": p, "Persistence_MAE": round(mae_p, 4),
                     "iTransformer_MAE": round(mae, 4),
                     "iTransformer_RMSE": round(rmse, 4) if rmse == rmse else np.nan,
                     "iTransformer_skill_%": round(skill, 1)})

metrics_df = pd.DataFrame(metrics).sort_values("iTransformer_skill_%", ascending=False).reset_index(drop=True)
metrics_df.insert(0, "rank", metrics_df.index + 1)
metrics_df.to_csv("metrics_itransformer_only.csv", index=False)
metrics_df


## 8. Reconstruct the 6 duplicate parameters from this model's forecast

Each duplicate is reconstructed as `slope * iTransformer's forecast of the kept twin + intercept`,
then scored against its own true (resampled) values on the held-out window — this is the
plot the 11-model notebook only summarized with one MAE number; here it gets its own
actual-vs-reconstructed comparison.

In [ ]:
dup_results = []
dup_series = {}
for keep, drop in DUPLICATES:
    keep_model, slope, intercept, r2_train = recon_coef[drop]
    true_drop = df_10min[drop].iloc[-HORIZON:]
    recon_pred = slope * pred_final[keep].values + intercept
    mae = mean_absolute_error(true_drop.values, recon_pred)
    rmse = np.sqrt(mean_squared_error(true_drop.values, recon_pred))
    dup_series[drop] = recon_pred
    dup_results.append({"duplicate_parameter": drop, "reconstructed_from": keep,
                         "slope": round(slope, 4), "intercept": round(intercept, 4),
                         "train_R2": round(r2_train, 5), "held_out_MAE": round(mae, 4),
                         "held_out_RMSE": round(rmse, 4)})
dup_df = pd.DataFrame(dup_results)
dup_df.to_csv("duplicate_reconstruction_itransformer_only.csv", index=False)
dup_df


## 9. Extract cross-variate attention weights

iTransformer's self-attention runs over the `n_features` variate-tokens. After training,
this re-runs the actual tokenization + first encoder layer's attention (with
`need_weights=True`, which the fast-path forward pass skips during normal training/
inference) on the test window, producing an `n_features × n_features` matrix where
`attention[i, j]` is how much parameter *i*'s forecast relies on parameter *j*.

In [ ]:
itransformer.eval()
with torch.no_grad():
    tok = itransformer.embed_tokens(last_window.to(device))
    # Layer 0's self-attention, with weights -- the fast (need_weights=False) path is what
    # the model actually uses during forward(); this is a parallel call purely for inspection.
    layer0 = itransformer.encoder.layers[0]
    normed = layer0.norm1(tok) if layer0.norm_first else tok
    _, attn_weights = layer0.self_attn(normed, normed, normed, need_weights=True, average_attn_weights=True)
    attn_weights = attn_weights[0].cpu().numpy()   # (n_features, n_features)

attn_df = pd.DataFrame(attn_weights, index=feature_cols, columns=feature_cols)
attn_df.to_csv("attention_weights_itransformer_only.csv")
print(f"Attention matrix: {attn_df.shape} (row = query parameter, column = key parameter it attends to)")

# Top-3 parameters each target parameter attends to most (excluding self-attention)
print("\nTop-3 attended-to parameters per target (excluding self):")
for p in target_cols[:8]:  # preview first 8
    row = attn_df.loc[p].drop(p, errors="ignore").sort_values(ascending=False).head(3)
    print(f"  {p:25s} -> {', '.join(f'{k} ({v:.3f})' for k, v in row.items())}")


## 10. Plot forecasts for all 24 parameters

In [ ]:
hist_tail = df_10min.iloc[-HORIZON - LOOKBACK:-HORIZON]   # raw cached data -- has circular cols in degrees, unlike df_num_full
key_plots = ["airTemperature", "tideLevel", "windSpeed", "significantWaveHeight",
             "currentSpeed", "salinity", "airPressure", "globalRadiation", "compass"]

fig, axes = plt.subplots(3, 3, figsize=(18, 11))
for ax, p in zip(axes.ravel(), key_plots):
    ax.plot(hist_tail.index, hist_tail[p], color="0.6", lw=1, label="history")
    ax.plot(truth.index, truth[p], color="black", lw=2.2, label="actual")
    ax.plot(truth.index, pred_final[p], color="tab:olive", lw=1.5, ls="--", label="iTransformer")
    ax.axvline(truth.index[0], color="green", lw=1, alpha=0.5)
    ax.set_title(p, fontsize=11); ax.grid(alpha=0.3)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
axes.ravel()[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Single iTransformer — 48h Forecast vs Actual (sample of 24 parameters)", fontsize=14, y=1.0)
fig.tight_layout()
fig.savefig("forecast_plots_itransformer_only.png", dpi=110, bbox_inches="tight")
plt.show()


## 11. Plot duplicate reconstructions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, (keep, drop) in zip(axes.ravel(), DUPLICATES):
    true_drop = df_10min[drop].iloc[-HORIZON:]
    hist_drop = df_10min[drop].iloc[-HORIZON - LOOKBACK:-HORIZON]
    ax.plot(hist_drop.index, hist_drop.values, color="0.6", lw=1, label="history")
    ax.plot(true_drop.index, true_drop.values, color="black", lw=2.2, label="actual")
    ax.plot(true_drop.index, dup_series[drop], color="tab:red", lw=1.5, ls="--",
            label=f"reconstructed (from {keep})")
    ax.axvline(true_drop.index[0], color="green", lw=1, alpha=0.5)
    ax.set_title(f"{drop}\n(R²={recon_coef[drop][3]:.4f} on train)", fontsize=10)
    ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30, labelsize=8)
axes.ravel()[0].legend(fontsize=8, loc="upper left")
fig.suptitle("6 Duplicate Parameters — Actual vs. Reconstructed from Kept Twin's iTransformer Forecast",
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("duplicate_reconstruction_plots_itransformer_only.png", dpi=110, bbox_inches="tight")
plt.show()


## 12. Plot the attention map

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(attn_df.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(feature_cols))); ax.set_xticklabels(feature_cols, rotation=90, fontsize=7)
ax.set_yticks(range(len(feature_cols))); ax.set_yticklabels(feature_cols, fontsize=7)
ax.set_xlabel("attends TO (key)"); ax.set_ylabel("query parameter")
ax.set_title("iTransformer Layer-0 Cross-Variate Attention Weights", fontsize=13)
fig.colorbar(im, ax=ax, label="attention weight")
fig.tight_layout()
fig.savefig("attention_map_itransformer_only.png", dpi=110, bbox_inches="tight")
plt.show()


## 13. Save outputs for the dashboard

In [ ]:
fva = pd.DataFrame({"timestamp": truth.index})
for p in report_params:
    fva[f"{p}__actual"] = truth[p].values
    fva[f"{p}__itransformer"] = pred_final[p].values
fva.to_csv("forecast_vs_actual_itransformer_only.csv", index=False)

dup_fva = pd.DataFrame({"timestamp": test_df.index})
for keep, drop in DUPLICATES:
    dup_fva[f"{drop}__actual"] = df_10min[drop].iloc[-HORIZON:].values
    dup_fva[f"{drop}__reconstructed"] = dup_series[drop]
dup_fva.to_csv("duplicate_forecast_vs_actual_itransformer_only.csv", index=False)

print("Saved: metrics_itransformer_only.csv, duplicate_reconstruction_itransformer_only.csv,")
print("       forecast_vs_actual_itransformer_only.csv, duplicate_forecast_vs_actual_itransformer_only.csv,")
print("       attention_weights_itransformer_only.csv, and 3 PNG plot files.")


## 14. How to read the results

- This is the **same iTransformer** that won 12/24 parameters in the 11-model
  comparison — isolating it here just gives it dedicated reconstruction and attention
  visualizations the bake-off notebook doesn't produce.
- **Duplicate reconstruction quality is bounded by two things**: how good the kept
  twin's forecast is, *and* how tight the linear relationship is (train R²). A duplicate
  can have a near-perfect R²=0.999 fit but still show reconstruction error on the
  held-out window if the kept twin's own 48h forecast has drifted from the truth — the
  reconstruction inherits 100% of the kept parameter's forecast error, by construction.
- **The attention map is a real interpretability tool, not just a visualization** — rows
  with concentrated attention on 1-2 other parameters indicate the model found a strong,
  specific dependency (e.g. air temperature attending heavily to dew point or humidity);
  rows with diffuse, near-uniform attention indicate the model is relying mostly on that
  parameter's own history (the per-parameter token's `var_id` and its own embedded
  lookback window dominate, with little cross-parameter signal contributing).
